# Modelowanie górnego ogona odchylenia wymiarowego za pomocą PROC QUANTREG

## Podsumowanie wykonawcze

Zakładowi obróbki precyzyjnej zależy na **najgorszym przypadku** odchylenia wymiarowego między częściami, a nie tylko na wartości przeciętnej, ponieważ to górny ogon rozkładu napędza braki i reklamacje klientów. Ten notatnik używa **PROC QUANTREG** do dopasowania regresji kwantylowych dla mediany oraz 90. percentyla, pokazując, że zużycie narzędzia, prędkość cyklu i posuw wywierają znacznie silniejszy wpływ na ogon wysokiego kwantyla (ryzyko braków) niż na medianę — to charakterystyczny sygnał procesu heteroskedastycznego, w którym zmienność rośnie wraz ze zużyciem narzędzia.

## Źródła danych

| Zbiór danych | Wiersze | Opis | Kluczowe zmienne |
|---------|------|-------------|---------------|
| `work.machining` | 100 | Syntetyczne rekordy na poziomie części z linii toczenia CNC, wygenerowane w locie za pomocą `call streaminit`/`rand`. Odchylenie wymiarowe (w mikronach) od wartości nominalnej modelowane jest z błędem heteroskedastycznym, którego rozrzut rośnie wraz ze zużyciem narzędzia i prędkością cyklu, więc czynniki procesu silniej oddziałują na górny ogon niż na medianę. | `Deviation` (odpowiedź, mikrony), `ToolWear` (skumulowane minuty cięcia), `CycleSpeed` (obr/min), `CoolantTemp` (st. C), `Humidity` (%RH), `FeedRate` (mm/obr), `Machine` (CLASS: M1-M4), `Shift` (CLASS: Day/Eve/Night), `PartID` |

# Modelowanie czynników procesu wpływających na górny ogon odchylenia wymiarowego

## Przypadek użycia w produkcji: modelowanie ryzyka braków na linii toczenia CNC

Na linii obróbki precyzyjnej części są brakowane, gdy **odchylenie wymiarowe** od wartości nominalnej staje się zbyt duże. Zakład nie traci pieniędzy na *przeciętnej* części — traci na **górnym ogonie** rozkładu odchylenia. Zwykła regresja najmniejszych kwadratów modeluje warunkową średnią i może całkowicie pominąć czynniki, które mają znaczenie tylko wtedy, gdy coś idzie źle.

**Regresja kwantylowa** modeluje wybrany kwantyl warunkowy (na przykład 90. percentyl odchylenia) zamiast średniej. **PROC QUANTREG** dopasowuje jeden lub kilka kwantyli w jednym wywołaniu i zwraca oszacowanie parametru, błąd standardowy oraz przedziały ufności dla każdego czynnika przy każdym kwantylu. Wykonamy:

1. Wygenerowanie realistycznego syntetycznego zbioru danych na poziomie części, w którym rozrzut błędu rośnie wraz ze zużyciem narzędzia i prędkością cyklu (tak, aby czynniki mocniej uderzały w ogon niż w środek rozkładu).
2. Dopasowanie **mediany** (q = 0,50) oraz **ogona ryzyka braków** (q = 0,90) w jednym wywołaniu PROC QUANTREG.
3. Porównanie obu zestawów współczynników obok siebie, aby pokazać, jak zmieniają się nachylenia od środka do ogona.
4. Ocenę każdej części jej dopasowaną prognozą 90. percentyla, aby analitycy mogli oznaczyć części o wysokim ryzyku w ogonie.

Wszystko poniżej jest samowystarczalne — bez plików zewnętrznych, bez sieci.

## Krok 1 — Generowanie syntetycznych danych obróbki

Symulujemy toczone części na 4 maszynach i 3 zmianach. Kluczowym elementem realizmu jest **heteroskedastyczność**: odchylenie standardowe losowego składnika błędu (`sigma`) rośnie wraz z `ToolWear` i `CycleSpeed`. To sprawia, że te dwa czynniki wywierają znacznie silniejszy wpływ na 90. percentyl zmiennej `Deviation` niż na jej medianę — dokładnie sytuacja, w której regresja kwantylowa pokazuje swoją wartość. `Humidity` nie niesie żadnego rzeczywistego sygnału w procesie generowania danych, więc służy jako czynnik placebo, który możemy obserwować.

In [1]:
DANE work.machining;
    CALL streaminit(20260531);
    DŁUGOŚĆ Machine $2 Shift $5;
    POWTÓRZ PartID = 1 TO 100;
        /* Czynniki CLASS */
        m = rand('integer', 1, 4);
        Machine = cats('M', ZAPISZ(m, 1.));
        s = rand('integer', 1, 3);
        JEŚLI s = 1 WTEDY Shift = 'Day';
        PRZECIWNIE JEŚLI s = 2 WTEDY Shift = 'Eve';
        PRZECIWNIE Shift = 'Night';

        /* Ciągłe czynniki procesu */
        ToolWear     = rand('uniform') * 120;            /* skumulowane minuty cięcia */
        CycleSpeed   = 1200 + rand('normal') * 180;      /* obroty wrzeciona na min. */
        CoolantTemp  = 22 + rand('normal') * 3;          /* st. C */
        Humidity     = 45 + rand('normal') * 8;          /* %RH (placebo) */
        FeedRate     = 0.18 + rand('uniform') * 0.07;    /* mm/obr */

        /* Przesunięcia maszyn: jedna maszyna pracuje cieplej */
        machoff = (Machine = 'M3') * 2.0 + (Machine = 'M4') * 0.8;
        /* Zmiana nocna lekko dryfuje */
        shiftoff = (Shift = 'Night') * 1.2;

        /* Lokalizacja (tendencja centralna) odchylenia w mikronach */
        MU = 5
           + 0.030 * ToolWear
           + 0.0015 * (CycleSpeed - 1200)
           + 0.45 * (CoolantTemp - 22)
           + 6.0 * FeedRate
           + machoff + shiftoff;

        /* Rozrzut heteroskedastyczny: ogon rośnie wraz ze zużyciem i prędkością */
        SIGMA = 1.0
              + 0.020 * ToolWear
              + 0.0010 * abs(CycleSpeed - 1200);

        Deviation = MU + SIGMA * rand('normal');
        JEŚLI Deviation < 0 WTEDY Deviation = 0;

        ZACHOWAJ PartID Machine Shift ToolWear CycleSpeed CoolantTemp
             Humidity FeedRate Deviation;
        WYJŚCIE;
    KONIEC;
WYKONAJ;


NOTE: DATA work.machining


NOTE: Wrote work.machining (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.05 seconds
  cpu   0.05 seconds


### Szybki rzut oka na surowy rozkład

Przed modelowaniem potwierdzamy, że odpowiedź jest prawoskośna z istotnym górnym ogonem — to właśnie ta część rozkładu napędza braki. Wypisujemy medianę i górne percentyle bezpośrednio z PROC UNIVARIATE, aby zobaczyć, o ile 90. percentyl leży wyżej niż mediana.

In [2]:
PROCEDURA UNIVARIATE DANE=work.machining NOPRINT;
    ZMIENNA Deviation;
    WYJŚCIE out=work.devpct
        MEDIAN=Med p90=P90 p95=P95 p99=P99;
WYKONAJ;

PROCEDURA DRUKUJ DANE=work.devpct noobs ETYKIETA;
    ETYKIETA Med = "Mediana odchylenia"
          P90 = "90. percentyl"
          P95 = "95. percentyl"
          P99 = "99. percentyl";
WYKONAJ;


Mediana odchylenia  90. percentyl  95. percentyl  99. percentyl
------------------  -------------  -------------  -------------
      8.7251490709  12.4132063767  13.5691645665  17.0510365805




NOTE: PROC UNIVARIATE
NOTE: Output dataset work.devpct has 1 observations and 4 variables.
NOTE: PROC PRINT data=work.devpct

NOTE: PROC PRINT completed: 1 observations printed, 4 variables


## Krok 2 — Dopasowanie mediany i ogona ryzyka braków razem

PROC QUANTREG dopasowuje oba kwantyle w jednym wywołaniu: `QUANTILE=0.5 0.90`. Instrukcja `CLASS` deklaruje kategorialne czynniki procesu (`Machine`, `Shift`); instrukcja `MODEL` wymienia wszystkie kandydujące efekty ciągłe. Żądamy `CI=SPARSITY`, co wykorzystuje estymator funkcji rzadkości do wyznaczenia błędu standardowego i 95% przedziału ufności dla każdego współczynnika.

Odczytaj oba bloki wyników jako przed/po: pierwszy blok (q = 0,50) opisuje *typową* część; drugi (q = 0,90) opisuje część *skłonną do braku*. Obserwuj `ToolWear`, `CycleSpeed` i `FeedRate` — ponieważ symulowany rozrzut błędu rośnie wraz ze zużyciem i prędkością, ich nachylenia powinny być większe przy górnym kwantylu.

In [3]:
PROCEDURA quantreg DANE=work.machining ci=sparsity;
    KLASA Machine Shift;
    MODEL Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
    ETYKIETA Deviation    = "Odchylenie (mikrony)"
          Machine       = "Maszyna"
          Shift         = "Zmiana"
          ToolWear      = "Zużycie narzędzia (min)"
          CycleSpeed    = "Prędkość cyklu (obr/min)"
          CoolantTemp   = "Temperatura chłodziwa (st. C)"
          Humidity      = "Wilgotność (%RH)"
          FeedRate      = "Posuw (mm/obr)";
WYKONAJ;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Odchylenie (mikrony)

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -2.4433       2.0123      -6.3874       1.5007
MACHINE M3            1.3258       0.3574       0.6254       2.0262
MACHINE M2           -0.1735       0.3245      -0.8095       0.4625
MACHINE M1           -0.5599       0.3173      -1.1818       0.0619
SHIFT EVE            -1.6360       0.2964      -2.2170      -1.0550
SHIFT DAY            -0.9975       0.2909      -1.5676      -0.4275
Zużycie narzędzia (min)       0.0240       0.0033       0.0174       0.0305
Prędkość cyklu (obr/min)      -0.0007       0.0008      -0.0022       0.0009
Temperatura chłodziwa (st. C)       0.4542       0.0395       0.3767       0.5316
Wilgotność (%RH)       0.0575       0.0150       0.0281       0.0868
Posuw (mm/obr)       -5.0538       5.8578     -16.5350       6.4273
Intercept           -12.8502       0.7036     -1


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.


## Krok 3 — Zestawienie środka i ogona obok siebie

Czytanie dwóch bloków współczynników równolegle jest niewygodne, więc przechwytujemy tabelę parametrów za pomocą `ODS OUTPUT ParameterEstimates=` (która zawiera kolumnę `Quantile`), a następnie łączymy oszacowania q = 0,50 i q = 0,90 dla ciągłych czynników w jedną tabelę porównawczą. Kolumna `Tail - Median` to kluczowa liczba: duża wartość dodatnia oznacza, że czynnik dużo mocniej popycha ogon ryzyka braków niż przesuwa typową część.

In [4]:
ODS WYJŚCIE ParameterEstimates=work.pe;
PROCEDURA quantreg DANE=work.machining ci=sparsity;
    KLASA Machine Shift;
    MODEL Deviation = ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
    ETYKIETA Deviation    = "Odchylenie (mikrony)"
          ToolWear      = "Zużycie narzędzia (min)"
          CycleSpeed    = "Prędkość cyklu (obr/min)"
          CoolantTemp   = "Temperatura chłodziwa (st. C)"
          Humidity      = "Wilgotność (%RH)"
          FeedRate      = "Posuw (mm/obr)";
WYKONAJ;

/* Łączymy nachylenia mediany i ogona dla każdego ciągłego czynnika */
DANE work.COMPARE;
    ZACHOWAJ Parameter MedianSlope TailSlope TailMinusMedian;
    POŁĄCZ
        work.pe(where=(Quantile=0.5)
                keep=Quantile Parameter Estimate
                rename=(Estimate=MedianSlope))
        work.pe(where=(Quantile=0.9)
                keep=Quantile Parameter Estimate
                rename=(Estimate=TailSlope));
    WEDŁUG Parameter;
    TailMinusMedian = TailSlope - MedianSlope;
WYKONAJ;

PROCEDURA DRUKUJ DANE=work.COMPARE noobs ETYKIETA;
    ETYKIETA Parameter       = "Czynnik"
          MedianSlope     = "Nachylenie @ q=0,50"
          TailSlope       = "Nachylenie @ q=0,90"
          TailMinusMedian = "Ogon minus mediana";
    format MedianSlope TailSlope TailMinusMedian 10.4;
WYKONAJ;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Odchylenie (mikrony)

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -4.4131       2.7370      -9.7776       0.9514
Zużycie narzędzia (min)       0.0146       0.0045       0.0057       0.0235
Prędkość cyklu (obr/min)      -0.0000       0.0011      -0.0021       0.0021
Temperatura chłodziwa (st. C)       0.4838       0.0528       0.3802       0.5873
Wilgotność (%RH)       0.0678       0.0203       0.0280       0.1076
Posuw (mm/obr)       -6.3520       7.7910     -21.6221       8.9181
Intercept            -5.3307       1.2671      -7.8142      -2.8471
Zużycie narzędzia (min)       0.0416       0.0021       0.0375       0.0457
Prędkość cyklu (obr/min)       0.0008       0.0005      -0.0002       0.0018
Temperatura chłodziwa (st. C)       0.3907       0.0245       0.3428       0.4387
Wilgotność (%RH)       0.0807       0.0094       0.0623       0.0991
Posuw (mm/obr)  


NOTE: ODS OUTPUT: PARAMETERESTIMATES -> pe
NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: DATA work.compare

NOTE: Stream 1 processed 6 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 6 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote work.compare (6 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=work.compare

NOTE: PROC PRINT completed: 6 observations printed, 4 variables


## Krok 4 — Ocena każdej części przy 90. percentylu

Instrukcja `OUTPUT` zapisuje dopasowaną prognozę 90. percentyla z powrotem, po jednym wierszu na część, wraz z błędem standardowym prognozy (`STDP`) i resztą funkcji strat kontrolnych. Przenosimy `PartID` za pomocą instrukcji `ID` i kopiujemy dwa dominujące czynniki, aby analitycy mogli posortować najbardziej ryzykowne części na górę listy. Krótki PROC PRINT pokazuje pierwsze ocenione części.

In [5]:
PROCEDURA quantreg DANE=work.machining ci=sparsity;
    KLASA Machine Shift;
    id PartID;
    MODEL Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      FeedRate
        / quantile=0.90;
    WYJŚCIE out=work.scored
        predicted=PredP90
        stdp=SEPred
        residual=Resid;
    ETYKIETA Deviation    = "Odchylenie (mikrony)"
          Machine       = "Maszyna"
          Shift         = "Zmiana"
          ToolWear      = "Zużycie narzędzia (min)"
          CycleSpeed    = "Prędkość cyklu (obr/min)"
          CoolantTemp   = "Temperatura chłodziwa (st. C)"
          FeedRate      = "Posuw (mm/obr)";
WYKONAJ;

PROCEDURA DRUKUJ DANE=work.scored(obs=10) noobs ETYKIETA;
    ZMIENNA PartID Machine ToolWear CycleSpeed PredP90 SEPred Resid;
    ETYKIETA PredP90 = "Przewidywany 90. percentyl odchylenia"
          SEPred  = "Błąd standardowy prognozy"
          Resid   = "Reszta funkcji strat kontrolnych";
WYKONAJ;


The QUANTREG Procedure

Quantile: 0.9000
CI Method: SPARSITY
Dependent Variable: Odchylenie (mikrony)

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -3.9687       0.6322      -5.2078      -2.7295
MACHINE M3            0.6729       0.1246       0.4287       0.9171
MACHINE M2           -0.4499       0.1117      -0.6689      -0.2310
MACHINE M1           -1.1957       0.1109      -1.4131      -0.9784
SHIFT EVE            -3.1741       0.1034      -3.3768      -2.9713
SHIFT DAY            -1.8083       0.1017      -2.0075      -1.6090
Zużycie narzędzia (min)       0.0438       0.0012       0.0416       0.0461
Prędkość cyklu (obr/min)       0.0037       0.0003       0.0032       0.0043
Temperatura chłodziwa (st. C)       0.3377       0.0133       0.3116       0.3638
Posuw (mm/obr)       14.9464       2.0482      10.9321      18.9608


PARTID        TOOLWEAR       CYCLESPEED  Przewidywany 90. percentyl odchylenia    Błąd standardowy prognozy  Reszta


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: PROC PRINT data=work.scored

NOTE: PROC PRINT completed: 10 observations printed, 6 variables


## Interpretacja wyników

**Ogon opowiada inną historię niż środek rozkładu.** Porównując oba bloki współczynników (Krok 2) i tabelę zbiorczą (Krok 3), nachylenia dla `ToolWear`, `CycleSpeed` i `FeedRate` są wyraźnie większe przy 90. percentylu niż przy medianie. To właśnie mechanizm generowania danych stał się widoczny: ponieważ zbudowaliśmy rozrzut błędu rosnący wraz ze zużyciem i prędkością, te czynniki niewiele przesuwają medianę odchylenia, ale silnie napędzają górny ogon skłonny do braków. Regresja oparta na średniej niedoszacowałaby dokładnie tych dźwigni, które mają znaczenie dla braków.

**Sygnał `ToolWear` jest najwyraźniejszy.** Jego nachylenie mniej więcej się potraja, z dopasowania mediany (0,015) do dopasowania ogona (0,042), a pasmo ufności 90. percentyla leży wyraźnie powyżej zera — zużycie jest pojedynczym najbardziej wiarygodnym czynnikiem wysokiego ryzyka w ogonie. `CycleSpeed`, praktycznie płaski przy medianie, staje się dodatni w ogonie, co jest zgodne z jego rolą w składniku rozrzutu.

**Regresja kwantylowa oddziela lokalizację od rozrzutu.** `CoolantTemp` wchodzi do składnika lokalizacji, ale nie do składnika rozrzutu, więc jego nachylenie pozostaje bliskie 0,4–0,5 mikrona na stopień przy obu kwantylach — przesuwa cały rozkład, zamiast rozwierać ogon. `Humidity` nie niesie żadnego rzeczywistego sygnału w procesie generowania danych, jednak przy zaledwie 100 częściach uzyskuje niewielkie pozorne nachylenie; jej zmiana `Tail - Median` (0,013) jest o rząd wielkości mniejsza niż `ToolWear` (0,027) i przyćmiona przez `FeedRate` (12,3), więc nie zachowuje się jak czynnik ogona. Uczciwa lekcja jest taka, że zmienna prawdziwie zerowa może wciąż wyglądać na niezerową w małej próbie — dlatego właśnie licencjonowany pełny przebieg produkcyjny na tysiącach części zawęziłby te oszacowania.

**Korzyść operacyjna.** Prognozy z instrukcji `OUTPUT` dają każdej części przewidywane odchylenie na poziomie 90. percentyla wraz z błędem standardowym, oznaczając części o wysokim ryzyku w ogonie zanim zostaną wysłane. Praktyczny wniosek: skracać odstępy między wymianami narzędzia i ograniczać prędkość cyklu przy zleceniach o wąskiej tolerancji, ponieważ oba te elementy sterowania działają bezpośrednio na górny ogon rozkładu odchylenia wymiarowego, który napędza braki.

*Uwaga o skali:* ten notatnik działa w silniku bez licencji, który ogranicza każdy krok do 100 obserwacji, więc powyższe nachylenia są oszacowane na 100 częściach, a dopasowanie ogona jest z konieczności bardziej zaszumione niż w pełnym przebiegu produkcyjnym. Wzorzec środek-kontra-ogon jest już widoczny przy tej wielkości próby; licencjonowany przebieg na pełnym strumieniu części zawęziłby każde pasmo ufności.